Imports

In [31]:
from time import thread_time_ns
import numpy as np
from pathlib import Path
import random
import itertools
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import cv2 as cv
from matplotlib import pyplot as plt
import numpy as np
import json
import math
import pprint
import pandas as pd
from time import thread_time_ns
from datetime import datetime
import statistics
from collections import defaultdict
import colorsys
from dataclasses import dataclass
from itertools import combinations
from math import hypot
import math
from PIL import Image
import seaborn as sns

Colorspace conversion functions

In [23]:
def rgb_to_luv(rgb):
        rgb = np.array([rgb[0], rgb[1], rgb[2]],dtype=np.uint8) # yellow
        rgb = rgb.reshape((1,1,3))
        luv = cv.cvtColor(rgb,cv.COLOR_RGB2Luv)[:,0][0]
        return (int(luv[0]), int(luv[1]), int(luv[2])) if(type(rgb)==tuple) else luv

def luv_to_rgb(luv):
    luv = np.array([luv[0], luv[1], luv[2]],dtype=np.uint8) # yellow
    luv = luv.reshape((1,1,3))
    rgb = cv.cvtColor(luv,cv.COLOR_LUV2RGB)[:,0][0]
    return (int(rgb[0]), int(rgb[1]), int(rgb[2])) if(type(luv)==tuple) else rgb

def rgb_to_cmyk(rgb):
    k = 1 - (max(rgb)/255)
    c = (1 - (rgb[0]/255)-k)/(1-k)
    m = (1 - (rgb[1]/255)-k)/(1-k)
    y = (1 - (rgb[2]/255)-k)/(1-k)
    return (round(c,3),round(m,3),round(y,3),round(k,3)) if type(rgb)==tuple else [round(c,3),round(m,3),round(y,3),round(k,3)]


def cmyk_to_rgb(cmyk):
     r= 255 * (1- cmyk[0]) * (1- cmyk[3])
     g= 255 * (1- cmyk[1]) * (1- cmyk[3])
     b= 255 * (1- cmyk[2]) * (1- cmyk[3])
     return [int(round(r,0)),int(round(g,0)),int(round(b,0))]

def cmyk_to_hsv(cmyk):
    rgb = cmyk_to_rgb(cmyk)
    return colorsys.rgb_to_hsv(rgb[0]/255, rgb[1]/255,rgb[2]/255)

def rgb_to_hsv(rgb):
    return colorsys.rgb_to_hsv(rgb[0]/255, rgb[1]/255,rgb[2]/255)

def hsv_to_rgb(hsv):
    rgb_scaled = colorsys.hsv_to_rgb(hsv[0], hsv[1], hsv[2])
    return (round(rgb_scaled[0]*255), round(rgb_scaled[1]*255),round(rgb_scaled[2]*255))

Class for points - used for finding rectangles

In [3]:
@dataclass
class Point:
    x: float = 0.0
    y: float = 0.0

    @classmethod
    def from_tuple(cls, coords):
        return cls(coords[0], coords[1])

    def as_tuple(self):
        return (self.x, self.y)


Functions associated with the class Point

In [4]:
def bucket(value: float, size: float):
    # print(value, int(value / size))
    return int(value / size)

def euc_dist(a: Point, b: Point) -> float:
    return hypot(b.x - a.x, b.y - a.y)

def same_midpoint(pairA, pairB, threshold=20):
    # note - threshold is in pixels
    def midpoint(pair):
        a, b = pair
        return Point((b.x - a.x) / 2.0, (b.y - a.y) / 2.0)

    # compute the midpoints of each pair
    midA = midpoint(pairA)
    midB = midpoint(pairB)

    # are they the same? or rather are they some delta distant
    distance = euc_dist(midA, midB)

    return distance < threshold


def point_as_ints(point):
    return int(point[0]), int(point[1])

def convertRectToList(rect):
    p0, p1, p2, p3 = rect
    p0, p2, p1, p3 = p0.as_tuple(), p2.as_tuple(), p1.as_tuple(), p3.as_tuple()
    return [point_as_ints(p) for p in [p0, p2, p1, p3]]


Functions for finding the clockwise dot relative to a single dot in a square

In [5]:
def angle_to(p, q):
    """Angle in radians from p to q, clockwise from horizontal right."""
    if p == q:
        raise ValueError("no angle from a point to itself")
    
    px, py = p
    qx, qy = q

    dx = qx - px
    dy = qy - py

    angle = math.atan(dy/dx)

    if dx < 0:
        angle += math.pi

    return angle


def clockwisePoint(points, black):
    """The point that is the one clockwise from black, in the square formed by points."""
    other_corners = [pt for pt in points if pt != black]
    dists_to_corners = [math.dist(black, q) for q in other_corners]
    furthest = max(dists_to_corners)
    opposite_corner = other_corners[dists_to_corners.index(furthest)]

    adjacent_corners = [pt for pt in other_corners if pt != opposite_corner]
    angles_to_corners = [angle_to(black, q) for q in adjacent_corners]

    angle_range = max(angles_to_corners) - min(angles_to_corners)

    if angle_range < math.pi:
        # should be about pi/2
        clockwise_angle = min(angles_to_corners)
    else:
        # should be about 3/2 pi
        clockwise_angle = max(angles_to_corners)

    clockwise_pt = adjacent_corners[angles_to_corners.index(clockwise_angle)]
    return clockwise_pt

Function to do clamping, ie bound a value.

In [6]:
def clamp(value, lowerbound, upperbound):
    if value < lowerbound:
        return lowerbound
    elif value > upperbound:
        return upperbound
    return value

## Detecting Circles on Paper

Functions to detect blobs, the first cell makes up a stepwise binary thresholder, which isn't currently being used, but good for a benchmark.
Second cell is a hough circle transform to recognize the dots, significantly faster.


In [7]:
def detection(thresholds,blob_params, gray):
    detector = cv.SimpleBlobDetector_create(blob_params)
    centers= []
    for thresh in thresholds:
        _, thresholded = cv.threshold(gray, thresh, 255, cv.THRESH_BINARY)
        keypoints = detector.detect(thresholded)
        current_centers = [(round(kp.pt[0]), round(kp.pt[1])) for kp in keypoints]
        new_centers = []
        for curr_center in current_centers:
            isNew = True
            for c in centers:
                    distance = math.dist(curr_center, c)
                    isNew = distance >= 10
                    if not isNew:
                        break
            if isNew:
                new_centers.append(curr_center)
        centers = centers + new_centers
    return centers


def blobParamFunc(minArea, minCircularity):
    params = cv.SimpleBlobDetector_Params()
    params.filterByCircularity = True
    params.minCircularity = minCircularity
    params.minArea=minArea
    params.blobColor = 0
    return params

def find_centers_mthresh(file_path):
    image = cv.imread(file_path, cv.IMREAD_COLOR_RGB)
    imageBlurred = cv.medianBlur(image, 3)
    gray = cv.cvtColor(imageBlurred, cv.COLOR_RGBA2GRAY)
    thresholds = np.arange(150,200,10)
    blob_params = blobParamFunc(400, .8)
    point_list = detection(thresholds=thresholds, blob_params= blob_params, gray=gray)
    return (point_list,image)
     

Two constants for calibrating and programs dots. Because the calibration grid dots are so close to each other, their distance is specific for that use case while the program min distance is taken from the hyper parameter search.

In [8]:
CALIBRATION_MIN_DIST = 30
PROGRAM_MIN_DIST = 170

In [9]:
def find_centers_hough(file_path, min_dist):
    image = cv.imread(file_path, cv.IMREAD_COLOR_RGB)
    imageBlurred = cv.medianBlur(image, 3)
    gray = cv.cvtColor(imageBlurred, cv.COLOR_RGBA2GRAY)
    circles = cv.HoughCircles(gray, cv.HOUGH_GRADIENT_ALT,1.5, min_dist, param1=300, param2=.9, minRadius=15, maxRadius=30)
    circles = circles[:,:,0:2][0] if circles is not None else []
    point_list = [(int(c[0]),int(c[1])) for c in circles]
    return (point_list,image)

## Finding the rectangles and squares made up by the circles

Get all __possible__ rectangles with ```get_all_rects()```


In [10]:
def get_all_rects(point_list):
    pointified = [Point.from_tuple(pt) for pt in point_list]
    pairs = list(combinations(pointified, 2))
    bucketed = [bucket(round(euc_dist(a, b)), 100) for a, b in pairs]
    tally = defaultdict(list)
    for pair, dist in zip(pairs, bucketed):
        tally[dist].append(pair)
    same_distance_pairs = dict(((dist, pairs) for dist, pairs in tally.items() if len(pairs) > 1))
    all_rects = []
    for _, pairs in same_distance_pairs.items():
        pairs_of_pairs = combinations(pairs, 2)
        rects = [(p1, p2) for (p1, p2) in pairs_of_pairs if same_midpoint(p1, p2)]
        all_rects.extend(rects)
    return all_rects

Get all __unique__ rectangles with ```check_rects()```

In [11]:
def check_rects(point_list):
    all_rects = get_all_rects(point_list)
    checked = []
    for rect in all_rects:
        p0, p2 = rect[0]
        p1, p3 = rect[1]
        rect_point_list = [p0, p2, p1, p3]
        target = Point(0,0)
        rect_point_list.sort(key= lambda p: euc_dist(p, target))
        if rect_point_list not in checked:
            checked.append(rect_point_list)
    return checked

Check which are squares  with ```is_square()```

In [12]:
def is_square(rect):
    [a,b,c,d] = convertRectToList(rect)
    vec_ab= np.array(a)-np.array(b)
    vec_ac = np.array(a)-np.array(c) 
    dist_ab = math.dist(a,b)
    dist_ac = math.dist(a,c)
    dot = np.dot(vec_ab, vec_ac)
    return (abs(dot) < 200 and abs(dist_ab- dist_ac) < 20)

Get squares from rectangle list with ```get_squares()```

In [13]:
def get_squares(rect_list):
    return [r for r in rect_list if is_square(r)]

For calibration purposes, return the big and the small squares relative to a point with ```small_big(squares,keypoint)```, specifically the black corner point<br>

The cal square looks like:<br>
. . .<br>
. . .<br>
. . .<br>

and labelled as 

a b c <br>
d e f <br>
g h i <br>

This will get you back<br>
b c <br>
e f <br>
as the small<br>

and <br>
a  c<br>

g  i <br>
as the big <br>

Calls ```rect_has``` to check that you are only grabbing the squares that have the keypoint, in calibration purposes, the black corner point

In [14]:
def rect_has(rect, dot):
        return dot in convertRectToList(rect)

def small_big(squares, keypoint):
    square_list = [rect for rect in squares if rect_has(rect,keypoint)]
    if len(square_list) > 2 or len(square_list) == 0:
        print("You have an empty list or too many squares, either way something is wrong!")
        return 
    square0 = square_list[0]
    square1 = square_list[1]
    [a0,b0,_,_] = convertRectToList(square0)
    [a1,b1,_,_] = convertRectToList(square1)
    dist0 = math.dist(a0,b0)
    dist1 = math.dist(a1,b1)
    if dist0 > dist1:
        small = square1
        big = square0
    else:
        small = square0
        big = square1
    return (small,big)

In [15]:
def displayFoundRect(rect, image):
    out = image.copy()
    p0,p1,p2,p3 = convertRectToList(rect)

    cv.line(out, p0, p2, (255, 0, 0), 10)
    cv.line(out, p0, p1, (255, 0, 0), 10)
    cv.line(out, p3, p1, (255, 0, 0), 10)
    cv.line(out, p3, p2, (255, 0, 0), 10)

    plt.imshow(out)
    plt.show()

## Color finding and identifying the encoding of the dots

```get_colors_and_coords``` returns the colors and the corresponding coordinates by grabbing the dots and the area around them using ```get_dot_list```. It uses this dot list to get the color the camera picks up for the circle and the "white" background that the circle has, and color corrects according to the colorspace chosen. RGB has the best results thus far.


Color Correcting Methods:


RGB - Because white (254,254,254) acts as a tail end of a range, 0-254, we can rescale the color found by whatever maximuim value is found for that channel so if the closest thing to white is (210, 250, 254), then the R channel would scale to a range of (0-210) by dividing the value by 210 and then multiplying by the result by 255 (to rescale to the 255 possible in the rgb channels), the green channel would divide by 250 and blue channel is untouched.

LUV - Because white is a point in euclidian space, we find the pixel closest to white, and then calculate the distance vector between that approximate white and what we know is "true" white in the color space. Then we apply that vector transform to every color found in the image.

CMYK and HSV have similar approaches to RGB but have not been investigated throughly as LUV and RGB seemed to be the most promising of colorspaces to color correct by.


In [16]:
def get_colors_and_coords(point_list, side, image, colorspace):
    def get_dot_list(point_list, side, image):
        dot_list = []
        for pt in point_list:
            x,y= pt
            crop_img = image[clamp(y-side, 0, image.shape[0]):clamp(y+side, 0, image.shape[0]), 
                             clamp(x-side, 0, image.shape[1]): clamp(x+side, 0, image.shape[1])]
            dot_list.append((crop_img, (x,y)))
        return dot_list

    dot_list = get_dot_list(point_list, side, image)

    colors_and_coords = []
    for dot_pair in dot_list:
        dot = dot_pair[0]
        x,y = dot_pair[1]
        maxPixel = (side*2)-1
        corner_points = [dot[0,maxPixel],dot[maxPixel,0],dot[0,maxPixel],dot[maxPixel,maxPixel]]
        if colorspace == "RGB":
            baseline_white_rgb = [max([corner_points[i][0] for i in range(0,4)]),
                            max([corner_points[i][1] for i in range(0,4)]),
                            max([corner_points[i][2] for i in range(0,4)])]
            ## white in this RGB is (255,255,255) so the maximal R G and B are the closest to white you will get
            circle_color= image[y,x]
            # print(f"circle color: {circle_color_rgb}")
            # print(f"baseline white: {baseline_white_rgb}")
            new = [clamp(round(circle_color[0]/baseline_white_rgb[0]*255),0,254),
                   clamp(round(circle_color[1]/baseline_white_rgb[1]*255),0,254),
                   clamp(round(circle_color[2]/baseline_white_rgb[2]*255),0,254)]
        elif colorspace == "HSV":
            corner_points_hsv = [rgb_to_hsv(p) for p in corner_points]
            baseline_white = [min([corner_points_hsv[i][0] for i in range(0,4)]),
                            min([corner_points_hsv[i][1] for i in range(0,4)]),
                            max([corner_points_hsv[i][2] for i in range(0,4)])]
            if baseline_white[0] ==0:
                baseline_white[0] = 1/179
            if baseline_white[1] ==0:
                baseline_white[1] = 1/255
            circle_color = rgb_to_hsv(image[y,x])
            # print(hsv_to_rgb(baseline_white))
            # print(f"circle color: {circle_color}")
            # print(f"baseline white: {baseline_white}")
            new = [clamp(round(circle_color[0]/baseline_white[0]*179),0,178)/179,
                   clamp(round(circle_color[1]/baseline_white[1]*255),0,254)/255,
                   clamp(round(circle_color[2]/baseline_white[2]*255),0,254)/255]
            ## When you convert, the white you grab for the white balance is different.
            ## thus the color correction will be different
            ## white in hsv is 0, 0, 100 so min min max
        elif colorspace == "CMYK":
            corner_points_cmyk = [rgb_to_cmyk(p) for p in corner_points]
            baseline_white = [min([corner_points_cmyk[i][0] for i in range(0,4)]),
                            min([corner_points_cmyk[i][1] for i in range(0,4)]),
                            min([corner_points_cmyk[i][2] for i in range(0,4)]),
                            min([corner_points_cmyk[i][3] for i in range(0,4)]),]
            circle_color = rgb_to_cmyk(image[y,x])
            # print(f"circle color: {circle_color}")
            # print(f"baseline white: {baseline_white}")
            new = [clamp(round(circle_color[0]-baseline_white[0],3),0,1),
                   clamp(round(circle_color[1]-baseline_white[1],3),0,1),
                   clamp(round(circle_color[2]-baseline_white[2],3),0,1),
                   clamp(round(circle_color[3]-baseline_white[3],3),0,1),]
            ## white in CMYK is 0, 0, 0, 0, so min min min min
        elif colorspace == "LUV":
            corner_points_luv = [rgb_to_luv(p) for p in corner_points]
            white_rgb = (255,255,255)
            white_luv = rgb_to_luv(white_rgb)
            min_dist = 10000
            baseline_white = None
            for pt in corner_points_luv:
                if math.dist(white_luv, pt) < min_dist:
                    min_dist = math.dist(white_luv, pt)
                    baseline_white = pt
            move = (int(baseline_white[0])-int(white_luv[0]), int(baseline_white[1])-int(white_luv[1]),int(baseline_white[2])-int(white_luv[2]))
            circle_color = rgb_to_luv(image[y,x])
            new = [clamp(round(int(circle_color[0])- move[0]),0,254),
                   clamp(round(int(circle_color[1])-move[1]),0,254),
                   clamp(round(int(circle_color[2])-move[2]),0,254)]
            # print(f"circle color in rgb pretransform: {image[y,x]}")
            print(f"circle color in luv pretransform: {circle_color}")
            print(f"baseline white: {baseline_white}")
            # print(f"circle color in rgb posttransform: {luv_to_rgb(new)}")
            print(f"circle color in luv posttransform: {new}")
        colors_and_coords.append((dot_pair[1], circle_color))
    return colors_and_coords

## CON

```get_black_dot``` gets the black dot by looking at a list of colors and their coordinates, and then transforming that color list into the colorspace HSV according to whatever colorspace was chosen first. If RGB, then convert to HSV. If HSV, no conversion. CMYK is not implemented. LUV, despite having a V channel, does not actually use the V similar to HSV, rather V and U create their own whitepoint, so it's more straightforward to go LUV->RGB->HSV. Once the color list is in the HSV space, it locates the lowest V channel color as the closest to "black" and then indexes into the original list to return the color coordinate pair for the black dot.

In [17]:
def get_black_dot(colors_and_coords, colorspace):
    colors_and_coords_hsv = []
    for c in colors_and_coords:
        if colorspace == "rgb":
            colors_and_coords_hsv.append((c[0],colorsys.rgb_to_hsv(c[1][0]/255, c[1][1]/255,c[1][2]/255)))
            values = [hsv[1][2] for hsv in colors_and_coords_hsv]
        elif colorspace == "hsv":
            values = [c[1][2] for c in colors_and_coords]
        elif colorspace == "cmyk":
            print(c)
        elif colorspace == "luv":
            # print(c)
            c_rgb = luv_to_rgb(c[1])
            colors_and_coords_hsv.append((c[0],colorsys.rgb_to_hsv(c_rgb[0]/255, c_rgb[1]/255,c_rgb[2]/255)))
            # print(colors_and_coords_hsv)
            values = [hsv[1][2] for hsv in colors_and_coords_hsv]
            # print(values)
    black_dot = (colors_and_coords[values.index(min(values))])
    return black_dot

```get_calibration_colors``` takes in the coordinates of the black dot and a color coordinate list from a calibration image. It uses the structure of the calibration grid, the ```get_squares``` and the ```small_big``` function to identify the first 6 dots relative to the black dot and then uses slope from point f to determine the identities of the remaining two dots. A dot that is closer to a parellel line with the point f to the line f h is point g and the other therefore must be e. It returns a dictionary of the colors found via the calibration image and their identifying letter.

In [18]:
def get_calibration_colors(black_dot_coords, color_coord_list):
    coord_list = [c[0] for c in color_coord_list]
    squares = get_squares(check_rects(coord_list))
    small, big = small_big(squares,black_dot_coords)
    small_list = convertRectToList(small)
    big_list = convertRectToList(big)
    calibration_order_list = [()] * 8
    if (black_dot_coords not in small_list) or (black_dot_coords not in big_list):
        print("Black Dot Not in at least one squares given, something is wrong!")
        return
    a = clockwisePoint(small_list, black_dot_coords)
    calibration_order_list[0] = a
    d = clockwisePoint(small_list, a)
    calibration_order_list[3] = d
    c = clockwisePoint(small_list, d)
    calibration_order_list[2] = c
    b = clockwisePoint(big_list, black_dot_coords)
    calibration_order_list[1]= b
    h = clockwisePoint(big_list, b)
    calibration_order_list[7] = h
    f = clockwisePoint(big_list, h)
    calibration_order_list[5] = f
    colored = [c for c in coord_list if c is not black_dot_coords]
    remaining = [ c for c in colored if c not in calibration_order_list]
    slope_fh = (f[1]-h[1]) / (f[0]-h[0])
    opt_1 = remaining[0]
    opt_2 = remaining[1]
    # print(len(colored))
    slope_f1 = (f[1]-opt_1[1]) / (f[0]-opt_1[0])
    slope_f2 = (f[1]-opt_2[1]) / (f[0]-opt_2[0])
    # print(f"abs(slope_fh - slope_f1)  {abs(slope_fh - slope_f1) }")
    # print(f"abs(slope_fh - slope_f2)  {abs(slope_fh - slope_f2) }")
    if abs(slope_fh - slope_f1) < abs(slope_fh - slope_f2):
        # print(f" abs(slope_fh - slope_f1) < abs(slope_fh - slope_f2) so g is ")
        calibration_order_list[6] = opt_1
        calibration_order_list[4] = opt_2
    else:
        calibration_order_list[4] = opt_1
        calibration_order_list[6] =opt_2
    calibration_order_list.insert(0,black_dot_coords)
    colored_coord_list = [c for c in color_coord_list if c[0] is not black_dot_coords]
    colors_sorted = sorted(colored_coord_list, key = lambda x: calibration_order_list.index(x[0]))
    colors = [c[1] for c in colors_sorted]
    # print(colors)
    letters = (list(map(chr, range(97, 105))))
    color_dict = {}
    for i in range(len(colors)):
        r,g,b = colors[i]
        color_dict.update({ (int(r),int(g),int(b)):letters[i]})
    return color_dict

```order_rectangle``` is built for program sheets, rather than calibration sheets, so it is only meant to get the clockwise order and thus used to find the encoding of a 4 dot rectangle.
It takes in a color and coordinate list and a reference point, typically the black dot previously found, but if a different dot is decided as the starting point, replace reference with that.
It returns a sorted version of the color and coordinate list in clockwise order. This is fed directly into ```get_color_perm```.

In [19]:
def order_rectangle(color_point_list, reference):
    point_list =[p[0] for p in color_point_list]
    ordered = [None, None, None]
    # print(reference)
    ordered[0] = clockwisePoint(point_list, reference)
    ordered[1] = clockwisePoint(point_list, ordered[0])
    # print(ordered)
    # print(point_list)
    ordered[2] = clockwisePoint(point_list, ordered[1])
    # ordered[2] = point_list[point_list not in ordered]
    # print(color_point_list)
    # print(ordered)
    return sorted(color_point_list, key = lambda x: ordered.index(x[0]))


```get_color_perm``` takes in an ordered rectangle in the form of a color coordinate list, a calibration dictionary and a choice of colorspace in which to calculate how close a given value is . Currently 'LUV' is the most accurate of the colorspaces for distance measurements. It returns a string which is the permutation of the dots that it thinks the rectangle is closest to.

In [20]:
def get_color_perm(ordered_rectangle, true_colors,printing:bool = False, colorspace:str = "LUV"):
    ordered_colors = [c[1] for c in ordered_rectangle]
    true_colors_custom_keys = true_colors.keys()
    perm = ""
    for col in ordered_colors:
        dist = 1000
        correct = ''
        for k in true_colors_custom_keys:
            if colorspace == "RGB":
                rgb_summed_diff = abs(k[0]-col[0]) +  abs(k[1]-col[1])  + abs(k[2]-col[2])
                if printing:
                    print(f"K in RGB is {k}")
                    print(f"Col in RGB is {col}")
                    print(f"The summed difference between {k}, and {col} is "+str(rgb_summed_diff) +"\n") 
                new_diff = rgb_summed_diff
            elif colorspace == "HSV":
                k_hsv = colorsys.rgb_to_hsv(k[0]/255, k[1]/255,k[2]/255)
                col_hsv = colorsys.rgb_to_hsv(col[0]/255, col[1]/255,col[2]/255)
                hsv_summed_diff = abs(k_hsv[0]-col_hsv[0]) +  abs(k_hsv[1]-col_hsv[1])  + abs(k_hsv[2]-col_hsv[2])
                if printing:
                    print(f"K in HSV is {k_hsv}")
                    print(f"Col in HSV is {col_hsv}")
                    print(f"The summed difference between {k_hsv}, and {col_hsv} is "+str(hsv_summed_diff) +"\n") 
                new_diff = hsv_summed_diff
            elif colorspace == "CMYK":
                k_cmyk = rgb_to_cmyk(k)
                col_cmyk = rgb_to_cmyk(col)
                cmyk_summed_diff= sum([abs((k_cmyk[0])-(col_cmyk[0])) ,  abs((k_cmyk[1])-(col_cmyk[1]))  , abs((k_cmyk[2])-(col_cmyk[2]))])
                if printing:
                    print(f"K in CMYK is {k_cmyk}")
                    print(f"Col in CMYK is {col_cmyk}")
                    print(f"The summed difference between {k_cmyk}, and {col_cmyk} is "+str(cmyk_summed_diff) +"\n") 
                new_diff = cmyk_summed_diff
            elif colorspace == "LUV":
                k_luv = rgb_to_luv(k)
                col_luv = rgb_to_luv(col)
                a =(int(k_luv[0])-int(col_luv[0]))**2
                b= (int(k_luv[1])-int(col_luv[1]))**2
                c =(int(k_luv[2])-int(col_luv[2]))**2
                pos = a+b+c
                luv_dist= math.sqrt(pos)
                if printing:
                    print(f"K in LUV is {k_luv}")
                    print(f"Col in LUV is {col_luv}")
                    print(f"The distance between {k_luv}, and {col_luv} is "+str(luv_dist) +"\n") 
                new_diff = luv_dist
            else:
                print("colorspace missing, please provide a choice")
            if new_diff < dist:
                dist = new_diff
                correct = true_colors.get(k)
                if printing:
                    print(new_diff)
                    print(f"the closest correct is {correct}")
        perm += correct
    return (perm)

## Evaluating on the dataset

First get the color dictionary via calibration

In [21]:
cal_path = "/Users/jdreiling/Desktop/puck/puck/images_calibration/high/custom/jack_cole/high_custom_jack_cole_calibration.jpg"
cal_centers,cal_image = find_centers_hough(cal_path, min_dist=CALIBRATION_MIN_DIST)
colors_and_coords=get_colors_and_coords(cal_centers,side = 25, image =cal_image,colorspace= "RGB")
black_dot_cal_coords = get_black_dot(colors_and_coords=colors_and_coords, colorspace= "rgb")[0]
calibration_colors =get_calibration_colors(black_dot_cal_coords,colors_and_coords)
calibration_colors

{(154, 48, 50): 'a',
 (134, 60, 123): 'b',
 (50, 127, 111): 'c',
 (36, 91, 194): 'd',
 (179, 153, 96): 'e',
 (153, 25, 84): 'f',
 (158, 198, 110): 'g',
 (65, 141, 175): 'h'}

Grab the paths of the dataset and set up a dictionary of the id, the correct permutation and the path list asocciated with the ID.

In [24]:
a_paths = []
b_paths = []
c_paths = []
d_paths = []

p = Path('..')

for path in p.glob("images/custom/*/*/A/*_A*[0-4].jpg"):
    a_paths.append("/Users/jdreiling/Desktop/puck/puck/" + str(path)[2:])


for path in p.glob("images/custom/*/*/B/*_B*[0-4].jpg"):
    b_paths.append("/Users/jdreiling/Desktop/puck/puck/" + str(path)[2:])

for path in p.glob("images/custom/*/*/C/*_C*[0-4].jpg"):
    c_paths.append("/Users/jdreiling/Desktop/puck/puck/" + str(path)[2:])

for path in p.glob("images/custom/*/*/D/*_D*[0-4].jpg"):
    d_paths.append("/Users/jdreiling/Desktop/puck/puck/" + str(path)[2:])

path_perm_dict = {"a": (a_paths, 'afd'),"b": (b_paths, 'ahc'),"c": (c_paths, 'ebg'),"d": (d_paths, 'fff') }


Check each image and see if it identifies the correct permutation. Note what permutation it finds and how much overlap there is with the true permutation. Put into a dataframe

In [27]:
checking_dict =[]
for k in path_perm_dict.keys():
    paths, true_perm = path_perm_dict.get(k)
    for path in paths:
        centers,image = find_centers_hough(path, min_dist=PROGRAM_MIN_DIST)
        colors_and_coords=get_colors_and_coords(centers,side = 25, image =image,colorspace= "RGB")
        if len(colors_and_coords) ==4:
            black_dot = get_black_dot(colors_and_coords=colors_and_coords, colorspace= "rgb")
            colored_coords = [c for c in colors_and_coords if c is not black_dot]
            # print(colored_coords)
            ordered_rectangle= order_rectangle(colored_coords, black_dot[0])
            perm = get_color_perm(ordered_rectangle, calibration_colors,colorspace= "LUV")
            correct = (perm == true_perm)
            overlap =sum([(perm[i] == true_perm[i]) for i in range(0,len(perm))])
        checking_dict.append({"key":k, "path":path, "found_perm": perm, "true_perm": true_perm, "correct": correct, "overlap":overlap})

results_df = pd.DataFrame(checking_dict, columns=["key","path", "found_perm", "true_perm", "correct", "overlap"])

Send that dataframe to a csv that will be stored in the results folder.

In [32]:
results_df.to_csv("../results/program_recognition_results.csv")


Color references for permutations and permutation sets.

In [ ]:
"#FD1622",a
"#FE00FF",b
"#00FE16",c
"#0084FE",d 
"#FEAA0D",e
"#DB4F89",f
"#EBFB26",g
"#00F8C4",h

A - '#FD1622', '#DB4F89', '#0084FE', afd
B - '#FD1622', '#00F8C4', '#00FE16',ahc
C - '#FEAA0D', '#DB4F89', '#EBFB26', ebg
D - "#3C1425", '#DB4F89', '#DB4F89' fff

